# scraping.ipynb - San Diego Nonprofit EIN Pull

"
"This notebook pulls San Diego nonprofits from the ProPublica Nonprofit Explorer API and saves clean CSV outputs.

"
"Notes:
"
"- Uses API endpoints documented by ProPublica (`/search.json` and `/organizations/:ein.json`).
"
"- Adds small delays between requests to be polite with rate limits.
"
"- Keeps EINs as zero-padded strings (`XXXXXXXXX`) for reliable joins.


In [1]:
%pip install requests pandas

import time
import requests
import pandas as pd

pd.set_option('display.max_columns', None)


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached charset_normalizer-3.4.5-cp312-cp312-macosx_10_13_universal2.whl.metadata (39 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached numpy-2.4.3-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
Using cached charset_normalizer-3.4.5-cp312-cp312-macosx_10_13_universal2.whl (280 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached urllib3-2.6.3-py3-none-any.whl (131 kB)
Using cached pandas-3.0.1-cp312-cp312-macosx_11_0_arm64.whl (9.9 MB)
Using cached certifi-2026.2.25-py3-none-any.whl (153 kB)
Using cached numpy-2.4.3-cp312-cp312-macosx_14_0_arm64.whl (5.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [pa

In [2]:
BASE = "https://projects.propublica.org/nonprofits/api/v2"

def search_nonprofits(q="san diego", state="CA", c_code=3, max_pages=40, sleep_s=0.2):
    """
    Pull paginated nonprofit search results.
    c_code=3 filters to 501(c)(3). Set to None to include all exempt types.
    """
    rows = []

    for page in range(max_pages):
        params = {"q": q, "page": page, "state[id]": state}
        if c_code is not None:
            params["c_code[id]"] = c_code

        r = requests.get(f"{BASE}/search.json", params=params, timeout=30)
        r.raise_for_status()
        payload = r.json()

        orgs = payload.get("organizations", [])
        if not orgs:
            break

        for o in orgs:
            rows.append({
                "ein": str(o.get("ein", "")).zfill(9) if o.get("ein") is not None else "",
                "strein": o.get("strein"),
                "name": o.get("name"),
                "sub_name": o.get("sub_name"),
                "city": o.get("city"),
                "state": o.get("state"),
                "ntee_code": o.get("ntee_code"),
                "subseccd": o.get("subseccd"),
                "score": o.get("score"),
                "search_query": payload.get("search_query"),
            })

        # stop once we hit the last page
        if page >= payload.get("num_pages", 1) - 1:
            break

        time.sleep(sleep_s)

    out = pd.DataFrame(rows).drop_duplicates(subset=["ein", "name", "city"], keep="first")
    return out

search_df = search_nonprofits(q="san diego", state="CA", c_code=3, max_pages=80, sleep_s=0.2)
print("rows:", len(search_df), "unique EINs:", search_df['ein'].nunique())
search_df.head(10)


rows: 1955 unique EINs: 1955


,ein,strein,name,sub_name,city,state,ntee_code,subseccd,score,search_query
0,820973253,82-0973253,San Diego Gaa,San Diego Gaa San Diego Gaelic Games,San Diego,CA,NaN,3,95.63512,san diego
1,260457477,26-0457477,Feeding San Diego,Feeding San Diego,San Diego,CA,K30,3,94.64624,san diego
2,953035557,95-3035557,Civic San Diego,Civic San Diego,San Diego,CA,S31Z,3,94.64624,san diego
3,330122462,33-0122462,Nami San Diego,Nami San Diego,San Diego,CA,Z99,3,94.64624,san diego
4,481291867,48-1291867,Kipp San Diego,Kipp San Diego,San Diego,CA,B29,3,94.64624,san diego
5,475534541,47-5534541,Alzheimers San Diego,Alzheimers San Diego,San Diego,CA,H83,3,94.64624,san diego
6,330647946,33-0647946,San Diego Coastkeeper,San Diego Coastkeeper,San Diego,CA,C320,3,94.64624,san diego
7,853336715,85-3336715,San Diego Squared,San Diego Squared,San Diego,CA,B82,3,94.64624,san diego
8,461740672,46-1740672,Ucp San Diego,Ucp San Diego,San Diego,CA,B20,3,94.64624,san diego
9,330461414,33-0461414,San Diego Ballet,San Diego Ballet,San Diego,CA,Z99Z,3,94.64624,san diego


In [4]:
def fetch_org_details(ein, sleep_s=0.15):
    """Pull organization profile + latest filing metrics for one EIN."""
    url = f"{BASE}/organizations/{int(ein)}.json"
    r = requests.get(url, timeout=30)
    if r.status_code != 200:
        return {
            "ein": str(ein).zfill(9),
            "org_city": None,
            "org_state": None,
            "org_zip": None,
            "subsection_code": None,
            "latest_tax_year": None,
            "latest_total_revenue": None,
            "latest_total_expenses": None,
            "latest_total_assets": None,
        }

    data = r.json()
    org = data.get("organization", {})
    filings = data.get("filings_with_data", []) or []

    latest = None
    if filings:
        latest = sorted(filings, key=lambda x: x.get("tax_prd", 0), reverse=True)[0]

    out = {
        "ein": str(org.get("ein", ein)).zfill(9),
        "org_city": org.get("city"),
        "org_state": org.get("state"),
        "org_zip": org.get("zipcode"),
        "subsection_code": org.get("subsection_code"),
        "latest_tax_year": latest.get("tax_prd_yr") if latest else None,
        "latest_total_revenue": latest.get("totrevenue") if latest else None,
        "latest_total_expenses": latest.get("totfuncexpns") if latest else None,
        "latest_total_assets": latest.get("totassetsend") if latest else None,
    }

    time.sleep(sleep_s)
    return out

# Pull details for unique EINs from search results
eins = search_df["ein"].dropna().astype(str).str.zfill(9).unique().tolist()

org_rows = []
total = len(eins)

for i, e in enumerate(eins, 1):
    org_rows.append(fetch_org_details(e))
    if i % 100 == 0 or i == total:
        print(f"{i}/{total} EINs processed ({i/total:.1%})")

org_df = pd.DataFrame(org_rows).drop_duplicates(subset=["ein"])

print("org detail rows:", len(org_df))
org_df.head(10)


100/1955 EINs processed (5.1%)
200/1955 EINs processed (10.2%)
300/1955 EINs processed (15.3%)
400/1955 EINs processed (20.5%)
500/1955 EINs processed (25.6%)
600/1955 EINs processed (30.7%)
700/1955 EINs processed (35.8%)
800/1955 EINs processed (40.9%)
900/1955 EINs processed (46.0%)
1000/1955 EINs processed (51.2%)
1100/1955 EINs processed (56.3%)
1200/1955 EINs processed (61.4%)
1300/1955 EINs processed (66.5%)
1400/1955 EINs processed (71.6%)
1500/1955 EINs processed (76.7%)
1600/1955 EINs processed (81.8%)
1700/1955 EINs processed (87.0%)
1800/1955 EINs processed (92.1%)


ReadTimeout: HTTPSConnectionPool(host='projects.propublica.org', port=443): Read timed out. (read timeout=30)

In [5]:
from requests.exceptions import ReadTimeout, ConnectionError, HTTPError

def fetch_org_details(ein, sleep_s=0.1, max_retries=3):
    """Pull organization profile + latest filing metrics for one EIN with retries."""
    ein_str = str(ein).zfill(9)
    url = f"{BASE}/organizations/{int(ein_str)}.json"

    for attempt in range(1, max_retries + 1):
        try:
            r = requests.get(url, timeout=30)
            r.raise_for_status()
            data = r.json()

            org = data.get("organization", {})
            filings = data.get("filings_with_data", []) or []
            latest = sorted(filings, key=lambda x: x.get("tax_prd", 0), reverse=True)[0] if filings else None

            time.sleep(sleep_s)
            return {
                "ein": str(org.get("ein", ein_str)).zfill(9),
                "org_city": org.get("city"),
                "org_state": org.get("state"),
                "org_zip": org.get("zipcode"),
                "subsection_code": org.get("subsection_code"),
                "latest_tax_year": latest.get("tax_prd_yr") if latest else None,
                "latest_total_revenue": latest.get("totrevenue") if latest else None,
                "latest_total_expenses": latest.get("totfuncexpns") if latest else None,
                "latest_total_assets": latest.get("totassetsend") if latest else None,
                "status": "ok",
            }

        except (ReadTimeout, ConnectionError, HTTPError):
            if attempt == max_retries:
                return {
                    "ein": ein_str,
                    "org_city": None,
                    "org_state": None,
                    "org_zip": None,
                    "subsection_code": None,
                    "latest_tax_year": None,
                    "latest_total_revenue": None,
                    "latest_total_expenses": None,
                    "latest_total_assets": None,
                    "status": "failed",
                }
            time.sleep(1.2 * attempt)  # backoff

# resume from where you left off (important)
done_eins = set(pd.DataFrame(org_rows)["ein"].astype(str).str.zfill(9)) if "org_rows" in globals() else set()
all_eins = search_df["ein"].dropna().astype(str).str.zfill(9).unique().tolist()
remaining = [e for e in all_eins if e not in done_eins]

print(f"Already done: {len(done_eins)} | Remaining: {len(remaining)}")

for i, e in enumerate(remaining, 1):
    org_rows.append(fetch_org_details(e))
    if i % 50 == 0 or i == len(remaining):
        print(f"{i}/{len(remaining)} remaining processed ({i/len(remaining):.1%})")

org_df = pd.DataFrame(org_rows).drop_duplicates(subset=["ein"])
print("org detail rows:", len(org_df))
print(org_df["status"].value_counts(dropna=False))


Already done: 1869 | Remaining: 86
50/86 remaining processed (58.1%)
86/86 remaining processed (100.0%)
org detail rows: 1955
status
NaN    1869
ok       86
Name: count, dtype: int64


In [6]:
# Combine search rows with org detail rows
propublica_sd = search_df.merge(org_df, on='ein', how='left')

# Keep only rows explicitly in San Diego city (if you want a strict city subset)
strict_sd = propublica_sd[
    propublica_sd['city'].astype(str).str.upper().eq('SAN DIEGO')
    | propublica_sd['org_city'].astype(str).str.upper().eq('SAN DIEGO')
].copy()

print('all rows from search:', len(propublica_sd))
print('strict SAN DIEGO city rows:', len(strict_sd))

propublica_sd.head(10)


all rows from search: 1955
strict SAN DIEGO city rows: 1409


,ein,strein,name,sub_name,city,state,ntee_code,subseccd,score,search_query,org_city,org_state,org_zip,subsection_code,latest_tax_year,latest_total_revenue,latest_total_expenses,latest_total_assets,status
0,820973253,82-0973253,San Diego Gaa,San Diego Gaa San Diego Gaelic Games,San Diego,CA,NaN,3,95.63512,san diego,San Diego,CA,92110-2767,3,NaN,NaN,NaN,NaN,NaN
1,260457477,26-0457477,Feeding San Diego,Feeding San Diego,San Diego,CA,K30,3,94.64624,san diego,San Diego,CA,92121-2934,3,2023.0,81172109.0,87467332.0,13391451.0,NaN
2,953035557,95-3035557,Civic San Diego,Civic San Diego,San Diego,CA,S31Z,3,94.64624,san diego,San Diego,CA,92108-1647,3,2024.0,37833041.0,36700224.0,22391966.0,NaN
3,330122462,33-0122462,Nami San Diego,Nami San Diego,San Diego,CA,Z99,3,94.64624,san diego,San Diego,CA,92123-1354,3,2023.0,12855737.0,12331113.0,5200798.0,NaN
4,481291867,48-1291867,Kipp San Diego,Kipp San Diego,San Diego,CA,B29,3,94.64624,san diego,San Diego,CA,92101-3245,3,2018.0,3929927.0,4191199.0,2654332.0,NaN
5,475534541,47-5534541,Alzheimers San Diego,Alzheimers San Diego,San Diego,CA,H83,3,94.64624,san diego,San Diego,CA,92123-1880,3,2023.0,2917722.0,3052762.0,4996290.0,NaN
6,330647946,33-0647946,San Diego Coastkeeper,San Diego Coastkeeper,San Diego,CA,C320,3,94.64624,san diego,San Diego,CA,92111-2111,3,2023.0,921217.0,1090517.0,2363056.0,NaN
7,853336715,85-3336715,San Diego Squared,San Diego Squared,San Diego,CA,B82,3,94.64624,san diego,San Diego,CA,92122-4449,3,2023.0,1319826.0,1001192.0,2879066.0,NaN
8,461740672,46-1740672,Ucp San Diego,Ucp San Diego,San Diego,CA,B20,3,94.64624,san diego,San Diego,CA,92103-2030,3,2024.0,1118520.0,1066528.0,720678.0,NaN
9,330461414,33-0461414,San Diego Ballet,San Diego Ballet,San Diego,CA,Z99Z,3,94.64624,san diego,San Diego,CA,92106-6172,3,2023.0,968733.0,969120.0,480101.0,NaN


In [7]:
# Optional: compare against local IRS extract already in repo
irs_path = 'irs_eo_san_diego.csv'
irs_sd = pd.read_csv(irs_path, low_memory=False)

# Try common EIN column names safely
possible_ein_cols = [c for c in irs_sd.columns if c.lower() in {'ein', 'einnum', 'ein_number'}]
if possible_ein_cols:
    ein_col = possible_ein_cols[0]
    irs_sd['ein'] = irs_sd[ein_col].astype(str).str.replace(r'\D', '', regex=True).str.zfill(9)
else:
    irs_sd['ein'] = ''

print('IRS rows:', len(irs_sd), 'IRS unique EINs:', irs_sd['ein'].nunique())
print('ProPublica unique EINs:', propublica_sd['ein'].nunique())
print('EIN overlap:', len(set(irs_sd['ein']) & set(propublica_sd['ein'])))


IRS rows: 16231 IRS unique EINs: 16231
ProPublica unique EINs: 1955
EIN overlap: 1649


In [8]:
# Save outputs
propublica_sd.to_csv('propublica_san_diego_nonprofits.csv', index=False)
strict_sd.to_csv('propublica_san_diego_city_only.csv', index=False)
org_df.to_csv('propublica_org_details_by_ein.csv', index=False)

print('Saved:')
print('- propublica_san_diego_nonprofits.csv')
print('- propublica_san_diego_city_only.csv')
print('- propublica_org_details_by_ein.csv')


Saved:
- propublica_san_diego_nonprofits.csv
- propublica_san_diego_city_only.csv
- propublica_org_details_by_ein.csv


In [31]:
propublica_sd.to_csv("/Users/jadenwu/Desktop/Move now/MOVE/propublica_SD_related_nonprofits.csv", index=False)
strict_sd.to_csv("/Users/jadenwu/Desktop/Move now/MOVE/propublica_SD_located_nonprofits.csv", index=False)
org_df.to_csv("/Users/jadenwu/Desktop/Move now/MOVE/propublica_org_details_by_ein.csv", index=False)

import os
for f in [
    "/Users/jadenwu/Desktop/Move now/MOVE/propublica_san_diego_nonprofits.csv",
    "/Users/jadenwu/Desktop/Move now/MOVE/propublica_san_diego_city_only.csv",
    "/Users/jadenwu/Desktop/Move now/MOVE/propublica_org_details_by_ein.csv",
]:
    print(os.path.exists(f), f)


False /Users/jadenwu/Desktop/Move now/MOVE/propublica_san_diego_nonprofits.csv
False /Users/jadenwu/Desktop/Move now/MOVE/propublica_san_diego_city_only.csv
True /Users/jadenwu/Desktop/Move now/MOVE/propublica_org_details_by_ein.csv


In [47]:
SD_scrape = pd.read_csv("propublica_SD_located_nonprofits.csv")
SD_scrape = SD_scrape.drop(columns=["state", 'org_city', 'org_state', 'search_query', 'org_zip', 'strein', 'score', 'subsection_code', 'status'])
SD_scrape

,ein,name,sub_name,city,ntee_code,subseccd,latest_tax_year,latest_total_revenue,latest_total_expenses,latest_total_assets
0,820973253,San Diego Gaa,San Diego Gaa San Diego Gaelic Games,San Diego,NaN,3,NaN,NaN,NaN,NaN
1,260457477,Feeding San Diego,Feeding San Diego,San Diego,K30,3,2023.0,81172109.0,87467332.0,13391451.0
2,953035557,Civic San Diego,Civic San Diego,San Diego,S31Z,3,2024.0,37833041.0,36700224.0,22391966.0
3,330122462,Nami San Diego,Nami San Diego,San Diego,Z99,3,2023.0,12855737.0,12331113.0,5200798.0
4,481291867,Kipp San Diego,Kipp San Diego,San Diego,B29,3,2018.0,3929927.0,4191199.0,2654332.0
...,...,...,...,...,...,...,...,...,...,...
1404,912100152,International Church Of The Foursquare Gospel,International Church Of The Foursquare Gospel ...,San Diego,NaN,3,NaN,NaN,NaN,NaN
1405,330078567,International Church Of The Foursquare Gospel,International Church Of The Foursquare Gospel ...,San Diego,NaN,3,NaN,NaN,NaN,NaN
1406,473156991,National Society Of Black Engineers,National Society Of Black Engineers Nsbe Profe...,San Diego,NaN,3,NaN,NaN,NaN,NaN
1407,330899235,Top Ladies Of Distinction Inc,Top Ladies Of Distinction Inc San Diego By The...,San Diego,NaN,3,NaN,NaN,NaN,NaN


In [37]:
irs = pd.read_csv("irs_eo_san_diego.csv")
irs_san_diego = irs[irs['CITY']== 'SAN DIEGO']
irs_san_diego

,EIN,NAME,ICO,STREET,CITY,STATE,ZIP,GROUP,SUBSECTION,AFFILIATION,CLASSIFICATION,RULING,DEDUCTIBILITY,FOUNDATION,ACTIVITY,ORGANIZATION,STATUS,TAX_PERIOD,ASSET_CD,INCOME_CD,FILING_REQ_CD,PF_FILING_REQ_CD,ACCT_PD,ASSET_AMT,INCOME_AMT,REVENUE_AMT,NTEE_CD,SORT_NAME
0,2296179,RELIGIOUS SCIENCE CHURCH CENTER OF SAN DIEGO,NaN,4102 MARLBOROUGH,SAN DIEGO,CA,92105-1462,0,3,3,7000,196204,1,10,1000000,1,1,NaN,0,0,6,0,3,NaN,NaN,NaN,NaN,NaN
2,10549309,ROBERTS FOUNDATION,% MARGARET GALLAGHER THOMPSON ESQ,PO BOX 9059,SAN DIEGO,CA,92169-0059,0,3,3,1000,200506,1,4,0,2,1,202412.0,7,4,0,1,12,8700413.0,407885.0,NaN,T22,NaN
4,10565671,SAN DIEGO RIVER PARK FOUNDATION,% ROB HUTSEL,4891 PACIFIC HWY STE 114,SAN DIEGO,CA,92110-4026,0,3,3,1000,200210,1,15,0,1,1,202412.0,8,6,1,0,12,32020724.0,4095868.0,4095868.0,N32,NaN
5,10573059,KAI ELUA OUTRIGGER CANOE CLUB,% ERIKA GAUDLITZ,1804 GARNET AVENUE SUITE 107,SAN DIEGO,CA,92109-3352,0,3,3,1200,201005,1,16,0,1,1,202410.0,3,3,1,0,10,31231.0,80939.0,80939.0,N67,A CALIFORNIA NON PROFIT CORPORATION
8,10635895,SHELTERCARE PROVIDERS OF SAN DIEGO INC,% JOHN PETERSON,PO BOX 927068,SAN DIEGO,CA,92192-7068,8137,3,9,1000,199510,0,15,994560000,1,1,202412.0,5,4,1,0,12,502267.0,427220.0,384245.0,NaN,HOMEAID SAN DIEGO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16221,995111690,GOOGLE GRANT SUCCESS HUB,% KERRY MENSIOR,10601 TIERRASANTA BLVD STE G-401,SAN DIEGO,CA,92124-2616,0,3,3,1000,202410,1,16,0,1,1,NaN,0,0,2,0,12,NaN,NaN,NaN,B60,NaN
16222,995127245,RISE & THRIVE PROGRAM,NaN,7517 DUNWOOD WAY,SAN DIEGO,CA,92114-7228,0,3,3,1000,202410,1,15,0,1,1,NaN,0,0,2,0,9,NaN,NaN,NaN,L41,NaN
16224,995136043,ROSIE PROJECT INC,NaN,10343 ROSELLE ST,SAN DIEGO,CA,92121-1501,0,3,3,1000,202410,1,16,0,1,1,202412.0,0,0,2,0,12,0.0,0.0,0.0,B90,NaN
16225,995137447,CPCMG CARES,NaN,3880 MURPHY CANYON RD STE 200,SAN DIEGO,CA,92123-4411,0,3,3,1000,202510,1,15,0,1,1,NaN,0,0,1,0,12,NaN,NaN,NaN,E60,NaN


Length of IRS + Publica Webscraped database is 7517

1223 of the non profits were merged together. Remaining are all independent

In [50]:
new = irs_san_diego.merge(SD_scrape, left_on = "EIN", right_on = "ein", how = 'outer', suffixes=("_a", '_b'), indicator=True)
new[new['_merge']=='both']
new.to_csv("San_Diego_Non_Profits_EIN.csv")
new

,EIN,NAME,ICO,STREET,CITY,STATE,ZIP,GROUP,SUBSECTION,AFFILIATION,CLASSIFICATION,RULING,DEDUCTIBILITY,FOUNDATION,ACTIVITY,ORGANIZATION,STATUS,TAX_PERIOD,ASSET_CD,INCOME_CD,FILING_REQ_CD,PF_FILING_REQ_CD,ACCT_PD,ASSET_AMT,INCOME_AMT,REVENUE_AMT,NTEE_CD,SORT_NAME,ein,name,sub_name,city,ntee_code,subseccd,latest_tax_year,latest_total_revenue,latest_total_expenses,latest_total_assets,_merge
0,2296179.0,RELIGIOUS SCIENCE CHURCH CENTER OF SAN DIEGO,NaN,4102 MARLBOROUGH,SAN DIEGO,CA,92105-1462,0.0,3.0,3.0,7000.0,196204.0,1.0,10.0,1000000.0,1.0,1.0,NaN,0.0,0.0,6.0,0.0,3.0,NaN,NaN,NaN,NaN,NaN,2296179.0,Religious Science Church Center Of San Diego,Religious Science Church Center Of San Diego,San Diego,NaN,3.0,NaN,NaN,NaN,NaN,both
1,10549309.0,ROBERTS FOUNDATION,% MARGARET GALLAGHER THOMPSON ESQ,PO BOX 9059,SAN DIEGO,CA,92169-0059,0.0,3.0,3.0,1000.0,200506.0,1.0,4.0,0.0,2.0,1.0,202412.0,7.0,4.0,0.0,1.0,12.0,8700413.0,407885.0,NaN,T22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,10565671.0,SAN DIEGO RIVER PARK FOUNDATION,% ROB HUTSEL,4891 PACIFIC HWY STE 114,SAN DIEGO,CA,92110-4026,0.0,3.0,3.0,1000.0,200210.0,1.0,15.0,0.0,1.0,1.0,202412.0,8.0,6.0,1.0,0.0,12.0,32020724.0,4095868.0,4095868.0,N32,NaN,10565671.0,San Diego River Park Foundation,San Diego River Park Foundation,San Diego,N32,3.0,2023.0,3004781.0,1359965.0,27779231.0,both
3,10573059.0,KAI ELUA OUTRIGGER CANOE CLUB,% ERIKA GAUDLITZ,1804 GARNET AVENUE SUITE 107,SAN DIEGO,CA,92109-3352,0.0,3.0,3.0,1200.0,201005.0,1.0,16.0,0.0,1.0,1.0,202410.0,3.0,3.0,1.0,0.0,10.0,31231.0,80939.0,80939.0,N67,A CALIFORNIA NON PROFIT CORPORATION,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,10635895.0,SHELTERCARE PROVIDERS OF SAN DIEGO INC,% JOHN PETERSON,PO BOX 927068,SAN DIEGO,CA,92192-7068,8137.0,3.0,9.0,1000.0,199510.0,0.0,15.0,994560000.0,1.0,1.0,202412.0,5.0,4.0,1.0,0.0,12.0,502267.0,427220.0,384245.0,NaN,HOMEAID SAN DIEGO,10635895.0,Sheltercare Providers Of San Diego Inc,Sheltercare Providers Of San Diego Inc Homeaid...,San Diego,NaN,3.0,2023.0,330889.0,315915.0,493557.0,both
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7512,995111690.0,GOOGLE GRANT SUCCESS HUB,% KERRY MENSIOR,10601 TIERRASANTA BLVD STE G-401,SAN DIEGO,CA,92124-2616,0.0,3.0,3.0,1000.0,202410.0,1.0,16.0,0.0,1.0,1.0,NaN,0.0,0.0,2.0,0.0,12.0,NaN,NaN,NaN,B60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7513,995127245.0,RISE & THRIVE PROGRAM,NaN,7517 DUNWOOD WAY,SAN DIEGO,CA,92114-7228,0.0,3.0,3.0,1000.0,202410.0,1.0,15.0,0.0,1.0,1.0,NaN,0.0,0.0,2.0,0.0,9.0,NaN,NaN,NaN,L41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7514,995136043.0,ROSIE PROJECT INC,NaN,10343 ROSELLE ST,SAN DIEGO,CA,92121-1501,0.0,3.0,3.0,1000.0,202410.0,1.0,16.0,0.0,1.0,1.0,202412.0,0.0,0.0,2.0,0.0,12.0,0.0,0.0,0.0,B90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
7515,995137447.0,CPCMG CARES,NaN,3880 MURPHY CANYON RD STE 200,SAN DIEGO,CA,92123-4411,0.0,3.0,3.0,1000.0,202510.0,1.0,15.0,0.0,1.0,1.0,NaN,0.0,0.0,1.0,0.0,12.0,NaN,NaN,NaN,E60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
